In [ ]:
# Download the citation attribution data
!mkdir -p data
!curl -sL -o data/citation_attribution_data.jsonl https://media.githubusercontent.com/media/lmarena/search-arena/main/data/citation_attribution_data.jsonl
!curl -sL -o arena_utils.py https://raw.githubusercontent.com/lmarena/search-arena/main/scripts/utils.py

In [3]:
import html
import pandas as pd

df = pd.read_json("data/citation_attribution_data.jsonl", lines=True)
df.shape

(780, 26)

In [13]:
df.winner.value_counts()

winner
model_b          269
model_a          266
tie              157
tie (bothbad)     88
Name: count, dtype: int64

In [4]:
# rebuild conversation
import html

def render(messages, trace):
    out, turn = [], 0
    for m in messages:
        text = html.unescape(m["content"])
        if m["role"] == "user":
            out.append(f"USER: {text}")
            continue
        refs = trace[turn] if turn < len(trace) else []
        ref_list = "\n".join(f"{marker} {url}" for marker, url in refs)
        out.append(f"ASSISTANT: {text}\n\nReferences:\n{ref_list}")
        turn += 1
    return "\n\n".join(out)

df["conv_a"] = [render(m, s["web_search_trace"]) for m, s in zip(df.messages_a, df.system_a_metadata)]
df["conv_b"] = [render(m, s["web_search_trace"]) for m, s in zip(df.messages_b, df.system_b_metadata)]
print(df.conv_a[0])

USER: most likely outcome in the upcoming match between fc gareji and dila gori

ASSISTANT: Based on recent performance, league standings, and head-to-head history, **Dila Gori is the favored team** to win the upcoming match against FC Gareji Sagarejo on April 24, 2025.

Here's a breakdown:

**Current Form & League Standing:**
*   **Dila Gori:** Holds the 3rd position in the Erovnuli Liga (Georgia's top tier) with 17 points from 9 matches. They have won 5, drawn 2, and lost 2 games this season[1][2]. Their recent form includes several wins and a strong away performance[1][2].
*   **FC Gareji:** Ranks 9th in the Erovnuli Liga with only 5 points from 9 matches. They have won just 1 game, drawn 2, and lost 6. They are currently on a losing streak in the league and have struggled to score, especially at home.

**Head-to-Head (H2H):**
*   The teams have played 5 times since 2023[2][3][4]. Gareji has won 3 matches and Dila Gori has won 2[2][3][8].
*   The most recent meeting was on April 1, 

In [5]:
import json, re
from pathlib import Path
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"

PROMPT = """You are comparing two AI assistants that can search the web. Both received the same user messages.

<conversation_A>
{a}
</conversation_A>

<conversation_B>
{b}
</conversation_B>

Which assistant responded better to the user? Choose one: A, B, tie, both_bad.
Reply only with JSON: {{"reasoning": "<one sentence>", "verdict": "A" | "B" | "tie" | "both_bad"}}"""

def judge(a, b):
    resp = client.messages.create(
        model=MODEL, max_tokens=200,
        messages=[{"role": "user", "content": PROMPT.format(a=a, b=b)}],
    )
    match = re.search(r"\{.*\}", resp.content[0].text, re.S)
    try:
        return json.loads(match.group(0))
    except (AttributeError, json.JSONDecodeError):
        return {"verdict": None, "reasoning": None}

flip = {"A": "B", "B": "A", "tie": "tie", "both_bad": "both_bad", None: None}
human = {"model_a": "A", "model_b": "B", "tie": "tie", "tie (bothbad)": "both_bad"}

In [25]:
out = Path("results/haiku.jsonl")
out.parent.mkdir(exist_ok=True)
done = {json.loads(line)["idx"] for line in out.open()} if out.exists() else set()

with out.open("a") as f:
    for idx, row in df.iterrows():
        if idx in done:
            continue
        fwd = judge(row.conv_a, row.conv_b)
        rev = judge(row.conv_b, row.conv_a)
        f.write(json.dumps({
            "idx": int(idx),
            "human": human[row.winner],
            "verdict_ab": fwd.get("verdict"),
            "verdict_ba": flip.get(rev.get("verdict")),
            "reason_ab": fwd.get("reasoning"),
            "reason_ba": rev.get("reasoning"),
        }) + "\n")
        f.flush()

In [6]:
res = pd.read_json("results/haiku.jsonl", lines=True)
len(res), res.verdict_ab.isna().sum(), res.verdict_ba.isna().sum()

(780, np.int64(0), np.int64(0))

In [7]:
res["consistent"] = res.verdict_ab == res.verdict_ba
res["judge"] = res.verdict_ab.where(res.consistent, "tie")

decisive = res[res.human.isin(["A", "B"])]

print("order consistency:", res.consistent.mean().round(3))
print("judge verdicts:\n", res.judge.value_counts(), "\n")
print("agreement with humans (decisive battles):", (decisive.judge == decisive.human).mean().round(3))

order consistency: 0.776
judge verdicts:
 judge
B      305
A      296
tie    179
Name: count, dtype: int64 

agreement with humans (decisive battles): 0.589


In [8]:
import numpy as np
from arena_utils import add_num_citations_style, add_cit_misattribution_counts, compute_bootstrap_style_control

model_map = {
    "gpt-4o-mini-search-preview": "gpt-4o-mini-search",
    "gpt-4o-mini-search-preview-high": "gpt-4o-mini-search-high",
    "gpt-4o-search-preview": "gpt-4o-search",
    "gpt-4o-search-preview-high": "gpt-4o-search-high",
    "gpt-4o-search-preview-high-loc": "gpt-4o-search-high-loc",
    "gemini-2.0-flash-grounding": "gemini-2.0-flash-grounding",
    "gemini-2.5-flash-preview-04-17-grounding": "gemini-2.5-flash-grounding",
    "gemini-2.0-pro-exp-02-05-grounding": "gemini-2.5-pro-grounding",
    "gemini-2.5-pro-exp-03-25-grounding": "gemini-2.5-pro-grounding",
    "gemini-2.5-pro-exp-03-25-wo-search": "gemini-2.5-pro",
}
FEATURES = ["support_count_a", "irrelevant_count_a", "contradict_count_a",
            "support_count_b", "irrelevant_count_b", "contradict_count_b"]

def citation_coefs(winner):
    d = df.drop(columns=["conv_metadata"]).copy()
    d["winner"] = winner.values
    d["model_a"] = d.model_a.replace(model_map)
    d["model_b"] = d.model_b.replace(model_map)
    d = add_num_citations_style(d, "citation_count")
    d = add_cit_misattribution_counts(d)
    d = d[(d.model_a != "gemini-2.5-pro") & (d.model_b != "gemini-2.5-pro")]
    d = d[d.conv_metadata.apply(lambda m: m["citation_count_a"] > 0 and m["citation_count_b"] > 0)]
    d = d[d.winner.isin(["model_a", "model_b"])]
    _, coefs = compute_bootstrap_style_control(d, style_elements=FEATURES, num_round=100)
    return len(d), coefs

judge_winner = (res.set_index("idx").judge.reindex(df.index)
                .map({"A": "model_a", "B": "model_b", "tie": "tie"}))

for label, w in [("human", df.winner), ("haiku", judge_winner)]:
    n, coefs = citation_coefs(w)
    lo, hi = np.percentile(coefs, [2.5, 97.5], axis=0)
    print(f"{label} (n={n})")
    for i, name in enumerate(["support", "irrelevant", "contradict"]):
        print(f"  {name:11} {coefs[:, i].mean():.3f}  [{lo[i]:.3f}, {hi[i]:.3f}]")

human (n=535)
  support     0.285  [0.035, 0.512]
  irrelevant  0.273  [0.050, 0.497]
  contradict  0.017  [-0.203, 0.202]
haiku (n=601)
  support     0.680  [0.462, 0.921]
  irrelevant  0.349  [0.093, 0.595]
  contradict  0.394  [0.180, 0.628]


In [9]:
from arena_utils import add_response_length_style

def citation_coefs(winner, features):
    d = df.drop(columns=["conv_metadata"]).copy()
    d["winner"] = winner.values
    d["model_a"] = d.model_a.replace(model_map)
    d["model_b"] = d.model_b.replace(model_map)
    d = add_num_citations_style(d, "citation_count")
    d = add_cit_misattribution_counts(d)
    d = add_response_length_style(d, "response_length")
    d = d[(d.model_a != "gemini-2.5-pro") & (d.model_b != "gemini-2.5-pro")]
    d = d[d.conv_metadata.apply(lambda m: m["citation_count_a"] > 0 and m["citation_count_b"] > 0)]
    d = d[d.winner.isin(["model_a", "model_b"])]
    _, coefs = compute_bootstrap_style_control(d, style_elements=features, num_round=100)
    return len(d), coefs

def show(label, winner, features):
    n, coefs = citation_coefs(winner, features)
    names = [f[:-2] for f in features[: len(features) // 2]]
    lo, hi = np.percentile(coefs, [2.5, 97.5], axis=0)
    print(f"{label} (n={n})")
    for i, name in enumerate(names):
        print(f"  {name:18} {coefs[:, i].mean():.3f}  [{lo[i]:.3f}, {hi[i]:.3f}]")
    return coefs

In [14]:
CIT = ["support_count_a", "irrelevant_count_a", "contradict_count_a",
       "support_count_b", "irrelevant_count_b", "contradict_count_b"]
CIT_LEN = ["support_count_a", "irrelevant_count_a", "contradict_count_a", "response_length_a",
           "support_count_b", "irrelevant_count_b", "contradict_count_b", "response_length_b"]

both_decisive = df.winner.isin(["model_a", "model_b"]) & judge_winner.isin(["model_a", "model_b"])

# reproduce the paper
_ = show("human", df.winner, CIT)

# length control
_ = show("human + length", df.winner, CIT_LEN)
_ = show("haiku + length", judge_winner, CIT_LEN)

human (n=535)
  support_count      0.285  [0.035, 0.512]
  irrelevant_count   0.273  [0.050, 0.497]
  contradict_count   0.017  [-0.203, 0.202]
human + length (n=535)
  support_count      0.116  [-0.142, 0.376]
  irrelevant_count   0.194  [-0.039, 0.410]
  contradict_count   -0.056  [-0.268, 0.132]
  response_length    0.615  [0.352, 0.861]
haiku + length (n=601)
  support_count      0.455  [0.190, 0.698]
  irrelevant_count   0.212  [-0.070, 0.448]
  contradict_count   0.291  [0.058, 0.548]
  response_length    1.360  [0.980, 1.815]


In [15]:
h = show("human, same battles", df.winner.where(both_decisive), CIT)
j = show("haiku, same battles", judge_winner.where(both_decisive), CIT)
np.percentile(j[:, 2] - h[:, 2], [2.5, 97.5])

human, same battles (n=426)
  support_count      0.258  [0.003, 0.537]
  irrelevant_count   0.329  [0.065, 0.569]
  contradict_count   0.036  [-0.188, 0.242]
haiku, same battles (n=426)
  support_count      0.581  [0.233, 0.880]
  irrelevant_count   0.368  [-0.043, 0.715]
  contradict_count   0.404  [0.158, 0.712]


array([0.11325543, 0.64349364])